# Notebook 01 — Data Acquisition & First Look

**What this notebook does:**
1. Mounts Google Drive so all outputs persist between sessions
2. Downloads the CFPB Consumer Complaint Database (full CSV)
3. Loads it **memory-safely** in chunks — avoids the RAM crash on free Colab
4. Inspects schema, targets, and narrative coverage
5. Plots class distributions
6. Takes a stratified 200k sample and saves it to Drive

**Why chunks?** The raw CSV is ~2 GB unzipped. Loading it all at once crashes free Colab (12 GB RAM limit). Instead we read 50k rows at a time, filter to narrative-only rows immediately, and only keep that smaller result in memory.

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# IMPORTANT: use the mounted filesystem path — NOT a Google Drive sharing URL
# After mounting, your Drive is accessible at /content/drive/MyDrive/
PROJECT_DIR = '/content/drive/MyDrive/nlp_project'
DATA_DIR    = os.path.join(PROJECT_DIR, 'data')
OUTPUT_DIR  = os.path.join(PROJECT_DIR, 'outputs')

os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Sanity check — these should start with /content/drive/MyDrive, never with https://
print(f'Project dir : {PROJECT_DIR}')
print(f'Data dir    : {DATA_DIR}')
print(f'Output dir  : {OUTPUT_DIR}')
assert PROJECT_DIR.startswith('/content/drive'), "Path looks wrong — should start with /content/drive, not a URL"
print('Drive mounted and folders ready.')

## Cell 2 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import zipfile
import urllib.request
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

print('Libraries loaded OK')

## Cell 3 — Download CFPB Data

Downloads to Colab's local disk (`/content/`) — faster than Drive for large files.
We only write the small 200k sample to Drive later.

**Takes ~3–5 minutes. Wait for it to finish before Cell 4.**

In [ ]:
DATA_URL = 'https://files.consumerfinance.gov/ccdb/complaints.csv.zip'
ZIP_PATH = '/content/complaints.csv.zip'
CSV_PATH = '/content/complaints.csv'

if not os.path.exists(CSV_PATH):
    print('Downloading (~500 MB) ...')
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print('Unzipping ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('Done.')
else:
    print('CSV already on disk, skipping download.')

print(f'Raw CSV size: {os.path.getsize(CSV_PATH)/1e6:.1f} MB')

## Cell 4 — Memory-Safe Load (chunked)

**This is the fix for the RAM crash.**

Instead of `pd.read_csv(file)` which loads 7M rows × 18 cols into RAM at once,
we read 50k rows at a time, normalise column names, filter to narrative-only rows,
and collect only those. The full CSV is never in memory simultaneously.

Result: `df_text` contains only the rows that have a complaint narrative — 
roughly 30–40% of the full dataset, small enough to work with comfortably.

**Takes ~4–6 minutes.**

In [ ]:
NARRATIVE_COL = 'consumer_complaint_narrative'
CHUNK_SIZE    = 50_000

chunks     = []
total_rows = 0
col_names  = None

print('Reading CSV in 50k-row chunks ...')

for i, chunk in enumerate(pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE, low_memory=False)):

    chunk.columns = (
        chunk.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_', regex=False)
        .str.replace('-', '_', regex=False)
    )

    if col_names is None:
        col_names = chunk.columns.tolist()

    total_rows += len(chunk)

    if NARRATIVE_COL in chunk.columns:
        # .astype(str) prevents AttributeError when pandas infers float64
        # on all-NaN chunks near end of file
        mask = chunk[NARRATIVE_COL].notna() & (chunk[NARRATIVE_COL].astype(str).str.strip() != '')
        if mask.any():
            chunks.append(chunk[mask])

    if (i + 1) % 30 == 0:
        kept = sum(len(c) for c in chunks)
        print(f'  Chunk {i+1:>4} | Rows processed: {total_rows:>8,} | Kept: {kept:>7,}')

df_text   = pd.concat(chunks, ignore_index=True)
n_with    = len(df_text)
n_without = total_rows - n_with

print(f'\nTotal rows in raw CSV    : {total_rows:,}')
print(f'Rows WITH narrative      : {n_with:,}  ({n_with/total_rows*100:.1f}%)')
print(f'Rows WITHOUT narrative   : {n_without:,}  ({n_without/total_rows*100:.1f}%)')
print(f'df_text RAM usage        : {df_text.memory_usage(deep=True).sum()/1e6:.0f} MB')
print(f'\nColumn names: {col_names}')

## Cell 5 — Confirm Key Columns Present

In [ ]:
KEY_COLS = [
    NARRATIVE_COL,
    'product', 'sub_product',
    'issue',   'sub_issue',
    'company', 'state', 'tags', 'submitted_via',
    'date_received', 'complaint_id'
]

present = [c for c in KEY_COLS if c in df_text.columns]
missing = [c for c in KEY_COLS if c not in df_text.columns]

print(f'Present ({len(present)}): {present}')
print(f'Missing ({len(missing)}): {missing}')

print('\n--- Sample rows (key cols) ---')
print(df_text[present].head(3).to_string())

## Cell 6 — Product Class Distribution (Primary Target)

In [ ]:
prod_counts = df_text['product'].value_counts()

print(f'Unique Product classes   : {prod_counts.shape[0]}')
print(prod_counts.to_string())
print(f'\nImbalance ratio (max/min): {prod_counts.max()/prod_counts.min():.1f}x')

fig, ax = plt.subplots(figsize=(10, 5))
prod_counts.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('CFPB — Product Distribution (narrative-only records)', fontsize=13)
ax.set_xlabel('Number of Complaints')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, 'product_distribution.png')
plt.savefig(fig_path, dpi=120)
plt.show()
print(f'Saved: {fig_path}')

## Cell 7 — Issue Class Distribution (Secondary Target)

In [ ]:
issue_counts = df_text['issue'].value_counts()

print(f'Unique Issue classes       : {issue_counts.shape[0]}')
print(f'Issues with < 500 records  : {(issue_counts < 500).sum()}')
print(f'Issues with < 100 records  : {(issue_counts < 100).sum()}')
print('\n--- Top 20 Issues ---')
print(issue_counts.head(20).to_string())

## Cell 8 — Narrative Length Distribution

Tells us how long complaint texts are — critical for deciding BERT's truncation strategy later.

In [ ]:
# str.count(' ') + 1 counts spaces instead of splitting into word lists
# avoids creating ~250M Python string objects in RAM (which kills the session)
df_text['narrative_len'] = df_text[NARRATIVE_COL].str.count(' ') + 1

print('--- Narrative word count stats ---')
print(df_text['narrative_len'].describe(
    percentiles=[.25, .5, .75, .90, .95, .99]
).to_string())

fig, ax = plt.subplots(figsize=(10, 4))
df_text['narrative_len'].clip(upper=600).hist(bins=80, ax=ax, color='coral', edgecolor='white')
ax.set_title('Complaint Narrative Length (words, clipped at 600)', fontsize=12)
ax.set_xlabel('Word Count')
ax.set_ylabel('Frequency')
plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, 'narrative_length_dist.png')
plt.savefig(fig_path, dpi=120)
plt.show()
print(f'Saved: {fig_path}')

## Cell 9 — Stratified Sample (200k rows)

Stratified by `product` so every class is proportionally represented in the sample.

In [ ]:
SAMPLE_SIZE = 200_000

df_text = df_text.dropna(subset=['product'])

frac = min(SAMPLE_SIZE / len(df_text), 1.0)
df_sample = (
    df_text
    .groupby('product', group_keys=False)
    .apply(lambda x: x.sample(frac=frac, random_state=42))
    .reset_index(drop=True)
)

print(f'Sample size: {len(df_sample):,}')
print('\nProduct distribution in sample:')
print(df_sample['product'].value_counts().to_string())

## Cell 10 — Save Sample to Google Drive

Every future notebook loads from this file. No need to re-download the raw data.

In [ ]:
KEEP_COLS = [
    'complaint_id', 'date_received',
    'product', 'sub_product',
    'issue',   'sub_issue',
    NARRATIVE_COL,
    'company', 'state', 'tags', 'submitted_via',
    'narrative_len'
]

keep     = [c for c in KEEP_COLS if c in df_sample.columns]
df_final = df_sample[keep].copy()

OUT_PATH = os.path.join(DATA_DIR, 'complaints_sample.csv')
df_final.to_csv(OUT_PATH, index=False)

# Verify save using the mounted path (never a URL)
assert OUT_PATH.startswith('/content/drive'), f"OUT_PATH looks wrong: {OUT_PATH}"
size_mb = os.path.getsize(OUT_PATH) / 1e6

print(f'Saved to    : {OUT_PATH}')
print(f'Rows        : {len(df_final):,}')
print(f'File size   : {size_mb:.1f} MB')
print(f'Columns     : {df_final.columns.tolist()}')

## Cell 11 — Summary (paste this output to Claude)

In [ ]:
print('='*58)
print('  NOTEBOOK 01 SUMMARY — paste this output to Claude')
print('='*58)
print(f'Total records in raw CSV          : {total_rows:,}')
print(f'Records WITH narrative            : {n_with:,} ({n_with/total_rows*100:.1f}%)')
print(f'Records WITHOUT narrative         : {n_without:,} ({n_without/total_rows*100:.1f}%)')
print(f'Sample size saved                 : {len(df_final):,}')
print(f'Sample CSV size (MB)              : {os.path.getsize(OUT_PATH)/1e6:.1f}')
print()
print('Product classes and counts (sample):')
print(df_final['product'].value_counts().to_string())
print()
print(f'Unique Issue classes              : {df_final["issue"].nunique()}')
print(f'Narrative word count — median     : {df_final["narrative_len"].median():.0f}')
print(f'Narrative word count — 95th pct   : {df_final["narrative_len"].quantile(0.95):.0f}')
print(f'Columns in saved file             : {df_final.columns.tolist()}')
print('='*58)